In [2]:
import pandas as pd

In [40]:
df=pd.read_csv("customer_shop.csv")
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [42]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [43]:
# write all columns in snakes cases 
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(' ','_')


In [44]:
df.columns
# df.head()

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [45]:
df["review_rating"]=df.groupby("category")["review_rating"].transform(lambda x:x.fillna(x.median()))

In [39]:
df.isnull().sum()

customer_id               0
age                       0
gender                    0
item_purchased            0
category                  0
purchase_amount_(usd)     0
location                  0
size                      0
color                     0
season                    0
review_rating             0
subscription_status       0
shipping_type             0
discount_applied          0
promo_code_used           0
previous_purchases        0
payment_method            0
frequency_of_purchases    0
Review Rating             0
dtype: int64

In [47]:
labels=["Young Adult","Adult","Middle_aged","Senior"]
df["age_group"]=pd.qcut(df["age"],q=4,labels=labels)

In [48]:
df[["age","age_group"]].head()

,age,age_group
0,55,Middle_aged
1,19,Young Adult
2,50,Middle_aged
3,21,Young Adult
4,45,Middle_aged


In [54]:
df["frequency_of_purchases"].head(25)


0        Fortnightly
1        Fortnightly
2             Weekly
3             Weekly
4           Annually
5             Weekly
6          Quarterly
7             Weekly
8           Annually
9          Quarterly
10         Bi-Weekly
11       Fortnightly
12       Fortnightly
13            Weekly
14            Weekly
15           Monthly
16         Bi-Weekly
17         Quarterly
18            Weekly
19         Bi-Weekly
20    Every 3 Months
21         Quarterly
22          Annually
23            Weekly
24            Weekly
Name: frequency_of_purchases, dtype: object

In [60]:
# pucheased frequency days columns

freq_mapping={
    "Fortnightly":14,
    "Weekly":7,
    "Annually":365,
    "Quarterly":90,
    "Bi-Weekly":14,
    "Every 3 Months":90,
    "Monthly":30
}



In [61]:
df["pucheased_feq_days"]=df["frequency_of_purchases"].map(freq_mapping)

In [62]:
df[["pucheased_feq_days","frequency_of_purchases"]].head()

,pucheased_feq_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually


In [66]:
df[["promo_code_used","discount_applied"]].head()

,promo_code_used,discount_applied
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes


In [68]:
(df["promo_code_used"]==df["discount_applied"]).all()  # to check if both columns are equal 

True

In [73]:
# df=df.drop("promo_code_used",axis=1)
df.columns


Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'pucheased_feq_days'],
      dtype='object')

In [75]:
#pip install psycopg2.binary sqlalchemy  ! add if pip install is not working 


In [77]:
#pip install pymysql sqlalchemy

In [80]:
from sqlalchemy import create_engine

# MySQL connection
username = "root"
password = "profitabletrader"
host = "localhost"
port = "3306"
database = "customer_behavior"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Write DataFrame to MySQL
table_name = "p_customer"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read back sample
# pd.read_sql("SELECT * FROM p_customer LIMIT 5;", engine)

,customer_id,age,gender,item_purchased,category,purchase_amount_(usd),location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,pucheased_feq_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle_aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle_aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle_aged,365
